# Boat detector v4s_frozen_v2 — expanded dataset

Train YOLOv8s with `freeze=10`, `imgsz=640`, and the expanded hard-example dataset. This starts a new run rather than resuming an older model. The historical bundle contains 3,944 training images, up from 2,802; verify the current dataset before training.

1. Choose a GPU under **Runtime > Change runtime type**.
2. Upload `boat_v4s_frozen_v2_bundle.zip` to Google Drive and update `ZIP_PATH`.
3. Run the following cells in order.


In [ ]:
!nvidia-smi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Set ZIP_PATH to the bundle you uploaded. Extraction resets /content/work.
ZIP_PATH = '/content/drive/MyDrive/boat_v4s_frozen_v2_bundle.zip'

!rm -rf /content/work
!mkdir -p /content/work
!unzip -q "$ZIP_PATH" -d /content/work
!ls /content/work


In [ ]:
!pip install -q ultralytics==8.4.138


In [ ]:
import pathlib

yaml_path = pathlib.Path('/content/work/yolo_dataset_v4/dataset.yaml')
content = yaml_path.read_text()
print('--- before ---')
print(content)

import yaml
settings = yaml.safe_load(content)
settings['path'] = '/content/work/yolo_dataset_v4'
new_content = yaml.safe_dump(settings, sort_keys=False)
yaml_path.write_text(new_content)
print('--- after ---')
print(yaml_path.read_text())


In [ ]:
from ultralytics import YOLO

_check = YOLO('/content/work/yolov8s.pt')
for i, layer in enumerate(_check.model.model):
    print(i, layer.__class__.__name__)


In [ ]:
# Start this experiment with the settings below.
from ultralytics import YOLO

model = YOLO('/content/work/yolov8s.pt')
results = model.train(
    data='/content/work/yolo_dataset_v4/dataset.yaml',
    epochs=80,
    imgsz=640,
    device=0,
    batch=16,
    patience=25,
    freeze=10,
    project='/content/work/runs_boat_yolo',
    name='boat_v4s_frozen_v2',
    verbose=True,
)


In [ ]:
# Resume from the latest checkpoint; retain optimizer and scheduler state.
# from ultralytics import YOLO
# model = YOLO('/content/work/runs_boat_yolo/boat_v4s_frozen_v2/weights/last.pt')
# results = model.train(resume=True)


In [ ]:
!mkdir -p /content/drive/MyDrive/boat_v4s_frozen_v2_results
!cp -r /content/work/runs_boat_yolo/boat_v4s_frozen_v2 /content/drive/MyDrive/boat_v4s_frozen_v2_results/
print('Copied to: Google Drive > boat_v4s_frozen_v2_results > boat_v4s_frozen_v2')


## Retrieve and evaluate the trained checkpoint

Download the result folder from the Google Drive destination printed above. Keep `weights/best.pt`, `weights/last.pt`, `args.yaml`, and `results.csv` together so checkpoint provenance is available. Pass the exact `best.pt` path to `run.py --yolo-weights` and evaluate it on independently reviewed recordings.

Colab sessions can disconnect. Keep recent checkpoints in Drive and resume from the latest `last.pt`; do not restart from the initial bundle and assume it contains newer training progress. Session duration and training speed are not guaranteed.
